# DOE MASTER ML PIPELINE — Equation-First V5 with occurrence labels and QC

This notebook runs a clean, equation-first gas-hydrate ML workflow for the four current wells.
It keeps the consolidated output design from V4 and adds a mentor-reviewable supervised occurrence-label demonstration.

**Scientific outputs**

1. Hydrate saturation regression: fixed complete-well validation using `WellA + WellB + WellC → blind validation on WellD`, then final refit on all labeled hydrate wells.
2. Water/residual-water saturation regression: `WellC → blind validation on WellD`, then final refit on available water-labeled wells.
3. Hydrate occurrence label rule: target-side occurrence labels are created from the approved hydrate saturation reference (`S_h`/`Sgh`/`Sh`) using explicit thresholds.
4. Hydrate occurrence classifier: a supervised classifier is trained on allowed log/equation features only, with the same fixed WellD holdout. The saturation-derived label is the `y` target and is blocked from predictors.
5. QC status: simple missing-log, caliper, elastic-validity, feature-coverage, and normalized-input flags are exported for review.

**Core idea**

Raw logs are standardized, equations are calculated first, target-derived equation fields are kept out of `X_allowed`, occurrence labels are separated from occurrence screens, and the ML benchmark uses train-only imputation/scaling under a complete-well split.

**Important runtime note**

The current workbook values are treated as normalized log inputs except depth. Equation-derived elastic/Archie fields are therefore first-run ML proxy features and calibration-review fields unless/until unnormalized physical-unit curves are supplied. Depth-derived hydrostatic pressure remains physical because it is calculated from depth.

## 0. Imports, paths, wells, and run policies

In [ ]:

from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable
import json
import math
import os
import re
import warnings

import joblib
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.base import clone
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor, RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression, Ridge, SGDRegressor
from sklearn.metrics import accuracy_score, balanced_accuracy_score, brier_score_loss, f1_score, mean_absolute_error, mean_squared_error, precision_score, r2_score, recall_score, roc_auc_score
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# -----------------------------------------------------------------------------
# Paths
# -----------------------------------------------------------------------------
DOWNLOADS = Path.home() / "Downloads"
DOE_INPUT_DIR = DOWNLOADS / "Northslopedatasets06052026"
SYNTH_INPUT_DIR = Path("/mnt/data/synth_north_slope")

if DOE_INPUT_DIR.exists():
    INPUT_DIR = DOE_INPUT_DIR
    OUTPUT_DIR = DOWNLOADS / "outputs_runtime" / "ml_master"
    MODEL_DIR = DOWNLOADS / "models_runtime" / "ml_master"
elif SYNTH_INPUT_DIR.exists():
    INPUT_DIR = SYNTH_INPUT_DIR
    OUTPUT_DIR = Path("/mnt/data/synth_workspace/outputs_runtime/ml_master")
    MODEL_DIR = Path("/mnt/data/synth_workspace/models_runtime/ml_master")
else:
    INPUT_DIR = DOE_INPUT_DIR
    OUTPUT_DIR = DOWNLOADS / "outputs_runtime" / "ml_master"
    MODEL_DIR = DOWNLOADS / "models_runtime" / "ml_master"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

# -----------------------------------------------------------------------------
# Well metadata and target map
# -----------------------------------------------------------------------------
# NOTE: latitude/longitude are kept as context only. They are not ML predictors.
# Site/well identity is used only to choose equation assumptions such as Archie a/m/n.
WELL_METADATA: dict[str, dict[str, Any]] = {
    "WellA": {
        "well_name": "Mallik 2L-38",
        "site": "Mallik / Mackenzie Delta analogue",
        "file_contains": ["2L-38"],
        "sheet": None,
        "layout": "mallik_stacked",
        "lat": np.nan,
        "lon": np.nan,
        "archie_a": np.nan,
        "archie_m": np.nan,
        "archie_n": np.nan,
        "archie_note": "Mallik uses NMR-density saturation as primary reference; Archie constants not forced unless approved.",
        "hydrate_target_header": "Sgh",
        "water_target_header": None,
    },
    "WellB": {
        "well_name": "Mallik 5L-38",
        "site": "Mallik / Mackenzie Delta analogue",
        "file_contains": ["5L-38"],
        "sheet": None,
        "layout": "mallik_stacked",
        "lat": np.nan,
        "lon": np.nan,
        "archie_a": np.nan,
        "archie_m": np.nan,
        "archie_n": np.nan,
        "archie_note": "Mallik uses NMR-density saturation as primary reference; Archie constants not forced unless approved.",
        "hydrate_target_header": "Sgh",
        "water_target_header": None,
    },
    "WellC": {
        "well_name": "Mount Elbert Stratigraphic Test Well",
        "site": "Mount Elbert / Eileen Gas Hydrate Trend",
        "file_contains": ["MtElbert", "Ignik", "ANS"],
        "sheet": "MTE",
        "layout": "direct_header",
        "lat": np.nan,
        "lon": np.nan,
        "archie_a": 1.0,
        "archie_m": 1.9,
        "archie_n": 2.0,
        "archie_note": "Alaska calibration window: Mount Elbert a=1.0, m=1.9, n=2.0.",
        "hydrate_target_header": "S_h",
        "water_target_header": "S_wr",
    },
    "WellD": {
        "well_name": "Iġnik Sikumi Test Well",
        "site": "Iġnik Sikumi / Eileen Gas Hydrate Trend",
        "file_contains": ["MtElbert", "Ignik", "ANS"],
        "sheet": "IGS",
        "layout": "direct_header",
        "lat": np.nan,
        "lon": np.nan,
        "archie_a": 1.6,
        "archie_m": 2.0,
        "archie_n": 2.0,
        "archie_note": "Alaska calibration window: Iġnik Sikumi a=1.6, m=2.0, n=2.0.",
        "hydrate_target_header": "Sh",
        "water_target_header": "Swr",
    },
}

# -----------------------------------------------------------------------------
# Equation constants and run policies
# -----------------------------------------------------------------------------
DENSITY_MATRIX_G_CC = 2.65
DENSITY_FLUID_G_CC = 1.02
SURFACE_PRESSURE_MPA = 0.101325
WATER_DENSITY_KG_M3 = 1020.0
GRAVITY_M_S2 = 9.80665

# Current data note from project review: workbook values are normalized except depth.
# Keep the original scientific names, but treat elastic/Archie outputs as first-run
# ML proxy features unless unnormalized physical-unit curves are later supplied.
NORMALIZED_INPUT_MODE = True

# Resistivity has been confirmed by project review as the deep formation resistivity family.
DEEP_RESISTIVITY_ALIAS_CONFIRMED = True
DEEP_RESISTIVITY_ALIASES_CONFIRMED = ["AO90", "A090", "AF90", "Deep formation resistivity", "Apparent Resistivity", "RES"]

# Caliper review: project notes say caliper is expected around 7.5-8.5 if physical
# diameter is present. If the caliper curve is normalized, the code marks it as
# normalized/context-only rather than failing the interval.
CALIPER_EXPECTED_MIN = 7.5
CALIPER_EXPECTED_MAX = 8.5

# If Rw is not supplied, estimate from S_wr/Swr where available.
# In normalized-input mode, this is a relative Archie-review proxy, not a final physical Rw.
ESTIMATE_RW_FROM_WATER_TARGET = True
RW_TRIM_QUANTILES = (0.05, 0.95)

# Fixed leave-one-well-out choice for the current mentor review: hold out WellD.
HYDRATE_TRAIN_WELLS = ["WellA", "WellB", "WellC"]
HYDRATE_VALIDATION_WELL = "WellD"

# Occurrence uses the same fixed complete-well split.
OCCURRENCE_TRAIN_WELLS = ["WellA", "WellB", "WellC"]
OCCURRENCE_VALIDATION_WELL = "WellD"
OCCURRENCE_POSITIVE_SH_THRESHOLD = 0.05
OCCURRENCE_NEGATIVE_SH_THRESHOLD = 0.01
OCCURRENCE_CLASSIFICATION_THRESHOLD = 0.50

# Water model: only wells with water target currently available.
WATER_TRAIN_WELLS = ["WellC"]
WATER_VALIDATION_WELL = "WellD"

MIN_TRAIN_FEATURE_COVERAGE = 0.35
MIN_ROW_FEATURE_FRACTION = 0.40

# Keep NMR porosity out of the default ML predictors because NMR-density saturation
# may be a target/calibration reference. The equation is still calculated for review.
INCLUDE_NMR_POROSITY_AS_FEATURE = False

print("RUN_ID:", RUN_ID)
print("INPUT_DIR:", INPUT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("MODEL_DIR:", MODEL_DIR)


## 1. Load workbooks and map raw headers to canonical fields

In [ ]:

# =============================================================================
# Workbook loading, standardization, and source provenance
# =============================================================================

def normalize_token(value: Any) -> str:
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass
    return "".join(ch for ch in str(value).strip().lower() if ch.isalnum())


def clean_text(value: Any, fallback: str = "") -> str:
    try:
        missing = pd.isna(value)
    except Exception:
        missing = False
    if missing:
        return fallback
    text = re.sub(r"\s+", " ", str(value).strip())
    return text or fallback


def numeric_series(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce")


def normalize_fraction(s: pd.Series) -> pd.Series:
    x = numeric_series(s)
    med = x.dropna().median() if x.notna().any() else np.nan
    if pd.notna(med) and med > 1.5:
        x = x / 100.0
    return x.clip(lower=0.0, upper=1.0)


def normalize_density_gcc(s: pd.Series) -> pd.Series:
    x = numeric_series(s)
    med = x.dropna().median() if x.notna().any() else np.nan
    # kg/m3 -> g/cc
    if pd.notna(med) and med > 20:
        x = x / 1000.0
    return x


def normalize_velocity_ms(s: pd.Series) -> pd.Series:
    x = numeric_series(s)
    med = x.dropna().median() if x.notna().any() else np.nan
    # km/s -> m/s
    if pd.notna(med) and 0.5 < med < 15:
        x = x * 1000.0
    return x


def find_input_file(tokens: list[str]) -> Path:
    files = [p for p in INPUT_DIR.glob("*.xls*") if not p.name.startswith("~$")]
    for token in tokens:
        for p in files:
            if token.lower() in p.name.lower():
                return p
    raise FileNotFoundError(f"Could not find workbook containing any of {tokens} under {INPUT_DIR}")


def read_mallik_stacked(path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    raw = pd.read_excel(path, sheet_name=0, header=None)
    if raw.shape[0] < 5:
        raise ValueError(f"{path.name} does not look like a stacked Mallik workbook")
    role_row = raw.iloc[0]
    header_row = raw.iloc[1]
    unit_row = raw.iloc[2]
    desc_row = raw.iloc[3]
    headers = []
    provenance_rows = []
    for i in range(raw.shape[1]):
        header = clean_text(header_row.iloc[i], f"column_{i}")
        if header.lower().startswith("unnamed") or header == "":
            header = clean_text(role_row.iloc[i], f"column_{i}")
        base = header
        count = sum(1 for h in headers if h == base or h.startswith(base + "."))
        if count:
            header = f"{base}.{count}"
        headers.append(header)
        provenance_rows.append({
            "source_column_index": i,
            "source_header": header,
            "role_row": clean_text(role_row.iloc[i]),
            "mnemonic_row": clean_text(header_row.iloc[i]),
            "unit_row": clean_text(unit_row.iloc[i]),
            "description_row": clean_text(desc_row.iloc[i]),
        })
    df = raw.iloc[4:].copy()
    df.columns = headers
    df = df.dropna(how="all").reset_index(drop=True)
    return df, pd.DataFrame(provenance_rows)


def read_direct_header(path: Path, sheet: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    df = pd.read_excel(path, sheet_name=sheet)
    df = df.dropna(how="all").reset_index(drop=True)
    prov = pd.DataFrame([
        {
            "source_column_index": i,
            "source_header": c,
            "role_row": "",
            "mnemonic_row": c,
            "unit_row": "",
            "description_row": "",
        }
        for i, c in enumerate(df.columns)
    ])
    return df, prov


def read_raw_well(alias: str, meta: dict[str, Any]) -> tuple[pd.DataFrame, pd.DataFrame, Path]:
    path = find_input_file(meta["file_contains"])
    if meta["layout"] == "mallik_stacked":
        df, prov = read_mallik_stacked(path)
    else:
        df, prov = read_direct_header(path, meta["sheet"])
    prov.insert(0, "well_alias", alias)
    prov.insert(1, "well_name", meta["well_name"])
    prov.insert(2, "source_workbook", path.name)
    prov.insert(3, "source_sheet", meta.get("sheet") or "first_sheet")
    return df, prov, path


def get_column(df: pd.DataFrame, aliases: Iterable[str]) -> tuple[pd.Series | None, str | None]:
    norm_to_col = {normalize_token(c): c for c in df.columns}
    aliases_norm = [normalize_token(a) for a in aliases]
    # exact first
    for a in aliases_norm:
        if a in norm_to_col:
            col = norm_to_col[a]
            return df[col], col
    # token containment second
    for a in aliases_norm:
        for norm, col in norm_to_col.items():
            if a and (a in norm or norm in a):
                return df[col], col
    return None, None


FIELD_ALIASES: dict[str, list[str]] = {
    "depth": ["depth_m", "depth", "true depth", "true_depth", "Depth_ft", "DEPT", "MD", "TVD"],
    "rhob_g_cc": ["rhob", "rho_b", "density_gpcc", "density", "bulk density", "Rho_b"],
    "density_porosity_vv": ["phi_den", "phi_porosity", "density porosity", "density_porosity", "DPHI", "PHID", "Phi_porosity"],
    "nmr_porosity_vv": ["phi_nmr", "nmr porosity", "NMR_POR", "CMR_PHI"],
    "neutron_porosity_vv": ["phi_neut", "neutron porosity", "NPHI", "TNPH"],
    "gr_api": ["gr", "gamma ray", "gamma_ray", "GR"],
    "rt_ohm_m": ["rt", "res", "resistivity", "deep formation resistivity", "A090", "AO90", "AF90", "ILD", "LLD"],
    "vp_m_s": ["vp", "velp", "compressional wave velocity", "p wave", "dtc_velocity"],
    "vs_m_s": ["vs", "vs1", "shear wave velocity", "s wave", "dts_velocity"],
    "caliper_raw": ["caliper", "cal1", "differential caliper", "cali"],
}


def standardize_well(alias: str, raw: pd.DataFrame, prov: pd.DataFrame, meta: dict[str, Any]) -> tuple[pd.DataFrame, pd.DataFrame]:
    out = pd.DataFrame(index=raw.index)
    mapping_rows = []

    def assign_field(field: str, transform=None):
        ser, col = get_column(raw, FIELD_ALIASES[field])
        if ser is None:
            out[field] = np.nan
            mapping_rows.append({"well_alias": alias, "canonical_field": field, "source_header": None, "coverage": 0.0, "note": "not_found"})
            return
        out[field] = transform(ser) if transform else numeric_series(ser)
        note = "mapped"
        if field == "rt_ohm_m" and DEEP_RESISTIVITY_ALIAS_CONFIRMED:
            note = "mapped_confirmed_deep_formation_resistivity"
        elif field == "caliper_raw":
            note = "mapped_caliper_qc_context"
        mapping_rows.append({
            "well_alias": alias,
            "canonical_field": field,
            "source_header": col,
            "coverage": float(out[field].notna().mean()),
            "note": note,
        })

    # Depth can be ft or m based on source header. Keep both for context.
    depth_ser, depth_col = get_column(raw, FIELD_ALIASES["depth"])
    if depth_ser is None:
        out["depth_m"] = np.nan
        out["depth_ft"] = np.nan
        mapping_rows.append({"well_alias": alias, "canonical_field": "depth_m", "source_header": None, "coverage": 0.0, "note": "not_found"})
    else:
        depth_raw = numeric_series(depth_ser)
        depth_norm = normalize_token(depth_col)
        if "ft" in depth_norm:
            out["depth_ft"] = depth_raw
            out["depth_m"] = depth_raw * 0.3048
            note = "converted_ft_to_m"
        else:
            out["depth_m"] = depth_raw
            out["depth_ft"] = depth_raw / 0.3048
            note = "assumed_m"
        mapping_rows.append({"well_alias": alias, "canonical_field": "depth_m", "source_header": depth_col, "coverage": float(out["depth_m"].notna().mean()), "note": note})

    assign_field("rhob_g_cc", normalize_density_gcc)
    assign_field("density_porosity_vv", normalize_fraction)
    assign_field("nmr_porosity_vv", normalize_fraction)
    assign_field("neutron_porosity_vv", normalize_fraction)
    assign_field("gr_api", numeric_series)
    assign_field("rt_ohm_m", numeric_series)
    assign_field("vp_m_s", normalize_velocity_ms)
    assign_field("vs_m_s", normalize_velocity_ms)
    assign_field("caliper_raw", numeric_series)

    # Targets by exact approved header map.
    h_ser, h_col = get_column(raw, [meta["hydrate_target_header"]])
    out["hydrate_saturation_reference"] = normalize_fraction(h_ser) if h_ser is not None else np.nan
    mapping_rows.append({"well_alias": alias, "canonical_field": "hydrate_saturation_reference", "source_header": h_col, "coverage": float(pd.Series(out["hydrate_saturation_reference"]).notna().mean()), "note": "target" if h_col else "not_found"})

    water_header = meta.get("water_target_header")
    if water_header:
        w_ser, w_col = get_column(raw, [water_header])
        out["water_saturation_reference"] = normalize_fraction(w_ser) if w_ser is not None else np.nan
        note = "target" if w_col else "not_found"
    else:
        w_col = None
        out["water_saturation_reference"] = np.nan
        note = "not_applicable"
    mapping_rows.append({"well_alias": alias, "canonical_field": "water_saturation_reference", "source_header": w_col, "coverage": float(pd.Series(out["water_saturation_reference"]).notna().mean()), "note": note})

    # Well metadata context.
    out.insert(0, "well_alias", alias)
    out.insert(1, "well_name", meta["well_name"])
    out.insert(2, "site", meta["site"])
    out["lat"] = meta.get("lat", np.nan)
    out["lon"] = meta.get("lon", np.nan)
    out["archie_a"] = meta.get("archie_a", np.nan)
    out["archie_m"] = meta.get("archie_m", np.nan)
    out["archie_n"] = meta.get("archie_n", np.nan)
    out["source_hydrate_header"] = meta["hydrate_target_header"]
    out["source_water_header"] = meta.get("water_target_header")

    # Clean invalid resistivity and velocities.
    out.loc[out["rt_ohm_m"] <= 0, "rt_ohm_m"] = np.nan
    out.loc[out["vp_m_s"] <= 0, "vp_m_s"] = np.nan
    out.loc[out["vs_m_s"] <= 0, "vs_m_s"] = np.nan

    return out, pd.DataFrame(mapping_rows)


def load_all_wells() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    frames = []
    mapping = []
    provenance = []
    for alias, meta in WELL_METADATA.items():
        raw, prov, path = read_raw_well(alias, meta)
        std, map_df = standardize_well(alias, raw, prov, meta)
        std["source_workbook"] = path.name
        std["source_sheet"] = meta.get("sheet") or "first_sheet"
        frames.append(std)
        mapping.append(map_df)
        provenance.append(prov)
        print(f"Loaded {alias}: {meta['well_name']} from {path.name}, rows={len(std)}")
    return pd.concat(frames, ignore_index=True), pd.concat(mapping, ignore_index=True), pd.concat(provenance, ignore_index=True)

standardized, field_mapping, source_columns = load_all_wells()
print("Combined rows:", len(standardized))
display(field_mapping)


## 2. Compute equation-derived fields before ML

In [ ]:

# =============================================================================
# Equation-first feature engineering
# =============================================================================

def clip01(x: pd.Series | np.ndarray | float) -> pd.Series | np.ndarray | float:
    return np.clip(x, 0.0, 1.0)


def safe_divide(numer, denom):
    return np.where(np.abs(denom) > 1e-12, numer / denom, np.nan)


def estimate_rw_by_well(df: pd.DataFrame) -> dict[str, float]:
    rw_by_well: dict[str, float] = {}
    for alias, group in df.groupby("well_alias"):
        meta = WELL_METADATA[alias]
        a = meta.get("archie_a", np.nan)
        m = meta.get("archie_m", np.nan)
        n = meta.get("archie_n", np.nan)
        if not np.isfinite(a) or not np.isfinite(m) or not np.isfinite(n):
            rw_by_well[alias] = np.nan
            continue
        if not ESTIMATE_RW_FROM_WATER_TARGET or group["water_saturation_reference"].notna().sum() < 10:
            rw_by_well[alias] = np.nan
            continue
        phi = group["density_porosity_vv"].copy()
        phi = phi.fillna(group.get("phi_density_calc", pd.Series(index=group.index, dtype=float)))
        rt = group["rt_ohm_m"]
        sw = group["water_saturation_reference"].clip(0.02, 0.98)
        rw = (sw ** n) * rt * (phi ** m) / a
        valid = rw.replace([np.inf, -np.inf], np.nan).dropna()
        valid = valid[(valid > 0) & (valid < valid.quantile(0.99) * 1.5)]
        if len(valid) >= 10:
            lo, hi = valid.quantile(RW_TRIM_QUANTILES[0]), valid.quantile(RW_TRIM_QUANTILES[1])
            valid = valid[(valid >= lo) & (valid <= hi)]
            rw_by_well[alias] = float(valid.median())
        else:
            rw_by_well[alias] = np.nan
    return rw_by_well


def add_equation_features(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    out = df.copy()
    equations = []

    # Hydrostatic pressure.
    pressure_gradient = WATER_DENSITY_KG_M3 * GRAVITY_M_S2 / 1_000_000.0
    out["pressure_hydrostatic_mpa"] = SURFACE_PRESSURE_MPA + pressure_gradient * out["depth_m"]
    equations.append({"field": "pressure_hydrostatic_mpa", "equation": "P_abs = P_surface + rho_w*g*z", "role": "context", "predictor_allowed": True, "notes": "Hydrostatic screening pressure only; not hydrate proof."})

    # Density porosity.
    out["phi_density_calc"] = (DENSITY_MATRIX_G_CC - out["rhob_g_cc"]) / (DENSITY_MATRIX_G_CC - DENSITY_FLUID_G_CC)
    out["phi_density_calc"] = out["phi_density_calc"].clip(0.0, 0.7)
    equations.append({"field": "phi_density_calc", "equation": "phi_D = (rho_matrix - rho_bulk) / (rho_matrix - rho_fluid)", "role": "feature", "predictor_allowed": True, "notes": f"rho_matrix={DENSITY_MATRIX_G_CC} g/cc; rho_fluid={DENSITY_FLUID_G_CC} g/cc."})

    # Use source density porosity when available, otherwise calculated density porosity.
    phi = out["density_porosity_vv"].copy()
    phi = phi.fillna(out["phi_density_calc"])
    out["phi_effective_for_equations"] = phi.clip(0.001, 0.7)
    equations.append({"field": "phi_effective_for_equations", "equation": "preferred phi = source density porosity else phi_D", "role": "feature", "predictor_allowed": True, "notes": "Used for Archie and physics features."})

    # NMR-density hydrate saturation.
    out["sh_nmr_density_calc"] = ((out["phi_effective_for_equations"] - out["nmr_porosity_vv"]) / out["phi_effective_for_equations"]).replace([np.inf, -np.inf], np.nan).clip(0.0, 1.0)
    equations.append({"field": "sh_nmr_density_calc", "equation": "Sh_NMRDEN = (phi - phi_NMR) / phi", "role": "target/calibration review", "predictor_allowed": False, "notes": "Kept out of X_allowed when learning hydrate saturation."})

    # GR shale volume proxy: simple linear index plus Larionov Tertiary form.
    gr_min = out["gr_api"].quantile(0.05)
    gr_max = out["gr_api"].quantile(0.95)
    if pd.isna(gr_min) or pd.isna(gr_max) or gr_max <= gr_min:
        gr_min, gr_max = 20.0, 120.0
    out["gr_index"] = ((out["gr_api"] - gr_min) / (gr_max - gr_min)).clip(0.0, 1.0)
    out["vsh_larionov_tertiary"] = (0.083 * (2 ** (3.7 * out["gr_index"]) - 1)).clip(0.0, 1.0)
    out["clean_sand_score"] = (1.0 - out["vsh_larionov_tertiary"]).clip(0.0, 1.0)
    equations.append({"field": "vsh_larionov_tertiary", "equation": "Vsh = 0.083*(2^(3.7*IGR)-1)", "role": "feature/context", "predictor_allowed": True, "notes": "Screening proxy using dataset 5th/95th GR anchors."})

    # Resistivity transform.
    out["log10_rt"] = np.log10(out["rt_ohm_m"].where(out["rt_ohm_m"] > 0))
    equations.append({"field": "log10_rt", "equation": "log10(Rt)", "role": "feature", "predictor_allowed": True, "notes": "Resistivity transform for ML."})

    # Elastic attributes.
    rho_kg_m3 = out["rhob_g_cc"] * 1000.0
    vp = out["vp_m_s"]
    vs = out["vs_m_s"]
    out["vp_vs_ratio"] = (vp / vs).replace([np.inf, -np.inf], np.nan)
    out["acoustic_impedance"] = out["rhob_g_cc"] * vp
    out["shear_impedance"] = out["rhob_g_cc"] * vs
    out["shear_modulus_gpa"] = (rho_kg_m3 * (vs ** 2)) / 1e9
    out["bulk_modulus_gpa"] = (rho_kg_m3 * ((vp ** 2) - (4.0 / 3.0) * (vs ** 2))) / 1e9
    out["youngs_modulus_gpa"] = (9 * out["bulk_modulus_gpa"] * out["shear_modulus_gpa"]) / (3 * out["bulk_modulus_gpa"] + out["shear_modulus_gpa"])
    out["poisson_ratio"] = ((vp ** 2) - 2 * (vs ** 2)) / (2 * ((vp ** 2) - (vs ** 2)))
    out["lambda_gpa"] = out["bulk_modulus_gpa"] - (2.0 / 3.0) * out["shear_modulus_gpa"]
    out["mu_gpa"] = out["shear_modulus_gpa"]
    out["lambda_rho"] = out["lambda_gpa"] * out["rhob_g_cc"]
    out["mu_rho"] = out["mu_gpa"] * out["rhob_g_cc"]
    out["lr_term"] = (out["acoustic_impedance"] ** 2) - 2 * (out["shear_impedance"] ** 2)
    out["mr_term"] = out["shear_impedance"] ** 2
    equations.extend([
        {"field": "vp_vs_ratio", "equation": "VpVs = Vp / Vs", "role": "feature", "predictor_allowed": True, "notes": "Elastic crossplot feature."},
        {"field": "acoustic_impedance", "equation": "AI = rho_bulk * Vp", "role": "feature", "predictor_allowed": True, "notes": "Stiffness-sensitive property."},
        {"field": "shear_modulus_gpa", "equation": "G = rho * Vs^2", "role": "feature", "predictor_allowed": True, "notes": "Computed in GPa."},
        {"field": "bulk_modulus_gpa", "equation": "K = rho*(Vp^2 - 4/3*Vs^2)", "role": "feature", "predictor_allowed": True, "notes": "Computed in GPa."},
        {"field": "youngs_modulus_gpa", "equation": "E = 9KG/(3K+G)", "role": "feature", "predictor_allowed": True, "notes": "Secondary elastic feature."},
        {"field": "poisson_ratio", "equation": "nu = (Vp^2 - 2Vs^2)/(2*(Vp^2 - Vs^2))", "role": "feature", "predictor_allowed": True, "notes": "Secondary elastic feature."},
        {"field": "lambda_rho", "equation": "lambda_rho = (K - 2/3G) * rho", "role": "feature", "predictor_allowed": True, "notes": "Fluid/incompressibility-sensitive elastic feature."},
        {"field": "mu_rho", "equation": "mu_rho = G * rho", "role": "feature", "predictor_allowed": True, "notes": "Rigidity-sensitive elastic feature."},
    ])

    # Rw estimation and Archie Sw/Sh baseline.
    rw_by_well = estimate_rw_by_well(out)
    out["rw_est_ohm_m"] = out["well_alias"].map(rw_by_well)
    out["sw_archie_calc"] = np.nan
    out["sh_archie_calc"] = np.nan
    for alias, meta in WELL_METADATA.items():
        idx = out["well_alias"] == alias
        a, m, n = meta.get("archie_a", np.nan), meta.get("archie_m", np.nan), meta.get("archie_n", np.nan)
        rw = rw_by_well.get(alias, np.nan)
        if np.isfinite(a) and np.isfinite(m) and np.isfinite(n) and np.isfinite(rw):
            phi_i = out.loc[idx, "phi_effective_for_equations"].clip(0.001, 0.7)
            rt_i = out.loc[idx, "rt_ohm_m"]
            sw = ((a * rw) / (rt_i * (phi_i ** m))) ** (1.0 / n)
            sw = sw.replace([np.inf, -np.inf], np.nan).clip(0.0, 1.0)
            out.loc[idx, "sw_archie_calc"] = sw
            out.loc[idx, "sh_archie_calc"] = (1.0 - sw).clip(0.0, 1.0)
    equations.append({"field": "sw_archie_calc", "equation": "Sw = ((a*Rw)/(Rt*phi^m))^(1/n)", "role": "physics baseline/review", "predictor_allowed": False, "notes": "Rw estimated from S_wr/Swr where available; not a predictor for saturation targets."})
    equations.append({"field": "sh_archie_calc", "equation": "Sh_Archie = 1 - Sw", "role": "physics baseline/review", "predictor_allowed": False, "notes": "Physics baseline/calibration comparison only."})

    # Occurrence screen: rule-based multi-log evidence, not supervised classifier.
    rt_score = ((out["log10_rt"] - np.log10(10)) / (np.log10(100) - np.log10(10))).clip(0.0, 1.0)
    vp_score = (((out["vp_m_s"] / 1000.0) - 2.5) / (4.0 - 2.5)).clip(0.0, 1.0)
    phi_score = ((out["phi_effective_for_equations"] - 0.15) / (0.35 - 0.15)).clip(0.0, 1.0)
    nmr_score = out["sh_nmr_density_calc"].clip(0.0, 1.0)
    archie_score = out["sh_archie_calc"].clip(0.0, 1.0)
    clean_score = out["clean_sand_score"].clip(0.0, 1.0)
    score_components = pd.DataFrame({
        "rt": rt_score,
        "vp": vp_score,
        "phi": phi_score,
        "clean": clean_score,
        "nmr": nmr_score,
        "archie": archie_score,
    })
    weights = pd.Series({"rt": 0.25, "vp": 0.20, "phi": 0.20, "clean": 0.15, "nmr": 0.10, "archie": 0.10})
    weighted = score_components.mul(weights, axis=1)
    available_weight = score_components.notna().mul(weights, axis=1).sum(axis=1)
    out["occurrence_probability_screen"] = (weighted.sum(axis=1) / available_weight).clip(0.0, 1.0)
    out.loc[available_weight < 0.35, "occurrence_probability_screen"] = np.nan

    def class_from_row(row):
        s = row["occurrence_probability_screen"]
        if pd.isna(s):
            return "insufficient_evidence"
        if s < 0.25:
            return "no_hydrate_or_water_like"
        if s < 0.45:
            return "possible_hydrate"
        if row.get("clean_sand_score", np.nan) < 0.35:
            return "ambiguous_resistive_or_stiff_anomaly"
        return "probable_pore_filling_hydrate"

    out["hydrate_occurrence_screen"] = out.apply(class_from_row, axis=1)
    equations.append({"field": "hydrate_occurrence_screen", "equation": "rule score from clean sand + Rt + Vp + phi + NMR/Archie evidence", "role": "rule-based occurrence screen", "predictor_allowed": False, "notes": "Independent log-evidence screen; not used as the supervised occurrence label."})

    # Supervised occurrence label rule: target-side label derived from approved hydrate saturation reference.
    # This is y for the occurrence classifier, never an X predictor.
    sh_ref = out["hydrate_saturation_reference"]
    out["hydrate_occurrence_label"] = np.nan
    out["occurrence_label_status"] = "missing_hydrate_saturation_reference"
    positive = sh_ref >= OCCURRENCE_POSITIVE_SH_THRESHOLD
    negative = sh_ref <= OCCURRENCE_NEGATIVE_SH_THRESHOLD
    gray = sh_ref.notna() & ~(positive | negative)
    out.loc[positive, "hydrate_occurrence_label"] = 1.0
    out.loc[positive, "occurrence_label_status"] = f"positive_Sh_ge_{OCCURRENCE_POSITIVE_SH_THRESHOLD:.2f}"
    out.loc[negative, "hydrate_occurrence_label"] = 0.0
    out.loc[negative, "occurrence_label_status"] = f"negative_Sh_le_{OCCURRENCE_NEGATIVE_SH_THRESHOLD:.2f}"
    out.loc[gray, "occurrence_label_status"] = f"gray_zone_{OCCURRENCE_NEGATIVE_SH_THRESHOLD:.2f}_to_{OCCURRENCE_POSITIVE_SH_THRESHOLD:.2f}_omitted"
    equations.append({"field": "hydrate_occurrence_label", "equation": f"label=1 if Sh >= {OCCURRENCE_POSITIVE_SH_THRESHOLD:.2f}; label=0 if Sh <= {OCCURRENCE_NEGATIVE_SH_THRESHOLD:.2f}; gray zone omitted", "role": "classification target", "predictor_allowed": False, "notes": "Mentor-review occurrence label rule derived from approved S_h/Sgh/Sh target only; blocked from X_allowed."})

    # Cleanup obvious impossible elastic values before QC.
    for col in ["vp_vs_ratio", "poisson_ratio", "bulk_modulus_gpa", "shear_modulus_gpa", "youngs_modulus_gpa", "lambda_rho", "mu_rho"]:
        out.loc[~np.isfinite(out[col]), col] = np.nan

    # QC and caveat flags. In normalized-input mode, these are review flags, not hard exclusions.
    raw_required = ["gr_api", "rhob_g_cc", "density_porosity_vv", "rt_ohm_m", "vp_m_s", "vs_m_s"]
    out["qc_required_log_coverage"] = out[[c for c in raw_required if c in out.columns]].notna().mean(axis=1)
    out["qc_low_required_log_coverage_flag"] = out["qc_required_log_coverage"] < 0.60
    out["qc_missing_rt_flag"] = out["rt_ohm_m"].isna()
    out["qc_missing_velocity_flag"] = out[["vp_m_s", "vs_m_s"]].isna().any(axis=1)
    out["qc_missing_porosity_flag"] = out[["density_porosity_vv", "phi_density_calc", "phi_effective_for_equations"]].isna().all(axis=1)

    cal = out["caliper_raw"]
    cal_med = cal.dropna().median() if cal.notna().any() else np.nan
    out["qc_caliper_status"] = "missing_caliper"
    if pd.notna(cal_med) and 2.0 <= cal_med <= 20.0:
        out["qc_caliper_status"] = np.where(cal.between(CALIPER_EXPECTED_MIN, CALIPER_EXPECTED_MAX), "physical_range_pass", "physical_range_review")
    elif pd.notna(cal_med):
        out["qc_caliper_status"] = "normalized_context_only"
    out["qc_bad_caliper_flag"] = out["qc_caliper_status"].eq("physical_range_review")

    out["qc_elastic_invalid_flag"] = (
        (out["vp_vs_ratio"] <= 1.0) |
        (out["poisson_ratio"] < -0.2) |
        (out["poisson_ratio"] > 0.5) |
        (out["shear_modulus_gpa"] <= 0) |
        (out["bulk_modulus_gpa"] <= 0)
    ).fillna(True)
    out["qc_normalized_input_mode"] = bool(NORMALIZED_INPUT_MODE)
    out["qc_deep_resistivity_alias_confirmed"] = bool(DEEP_RESISTIVITY_ALIAS_CONFIRMED)
    out["qc_status"] = np.where(
        out[["qc_low_required_log_coverage_flag", "qc_missing_rt_flag", "qc_missing_velocity_flag", "qc_missing_porosity_flag", "qc_bad_caliper_flag"]].any(axis=1),
        "review",
        "pass"
    )
    equations.append({"field": "qc_status", "equation": "review if low coverage, missing Rt/velocity/porosity, or physical caliper outside expected range", "role": "QC/caveat", "predictor_allowed": False, "notes": "All QC flags are exported for mentor review; normalized-input mode prevents physical overclaiming."})
    equations.append({"field": "normalized_input_mode", "equation": "all non-depth workbook curves treated as normalized inputs", "role": "runtime caveat", "predictor_allowed": "n/a", "notes": "Elastic and Archie outputs are first-run proxy/calibration features until unnormalized physical-unit logs are supplied."})

    eq_df = pd.DataFrame(equations)
    return out, eq_df

features_df, equations_used = add_equation_features(standardized)
print("Equation features added:", len(features_df.columns))
print("Rw estimates:")
display(features_df[["well_alias", "well_name", "rw_est_ohm_m"]].drop_duplicates())
print("Occurrence screen counts:")
display(features_df["hydrate_occurrence_screen"].value_counts(dropna=False).to_frame("count"))
print("Occurrence label counts from S_h rule:")
display(features_df[["occurrence_label_status", "hydrate_occurrence_label"]].value_counts(dropna=False).to_frame("rows"))
print("QC status counts:")
display(features_df["qc_status"].value_counts(dropna=False).to_frame("rows"))


## 3. Train models under fixed complete-well validation

In [ ]:
# =============================================================================
# ML models — fixed blind-well validation, then final refit
# =============================================================================

BASE_FEATURES = [
    "gr_api",
    "rhob_g_cc",
    "density_porosity_vv",
    "neutron_porosity_vv",
    "rt_ohm_m",
    "log10_rt",
    "vp_m_s",
    "vs_m_s",
    "vp_vs_ratio",
    "acoustic_impedance",
    "shear_impedance",
    "shear_modulus_gpa",
    "bulk_modulus_gpa",
    "youngs_modulus_gpa",
    "poisson_ratio",
    "lambda_rho",
    "mu_rho",
    "phi_density_calc",
    "phi_effective_for_equations",
    "vsh_larionov_tertiary",
    "clean_sand_score",
    "pressure_hydrostatic_mpa",
]
if INCLUDE_NMR_POROSITY_AS_FEATURE:
    BASE_FEATURES.append("nmr_porosity_vv")

# Direct target-derived fields and review outputs are deliberately excluded from X_allowed.
BLOCKED_FEATURES = {
    "hydrate_saturation_reference",
    "water_saturation_reference",
    "sh_nmr_density_calc",
    "sw_archie_calc",
    "sh_archie_calc",
    "occurrence_probability_screen",
    "hydrate_occurrence_screen",
    "hydrate_occurrence_label",
    "occurrence_label_status",
    "qc_status",
}

REGRESSION_MODELS = {
    "mean_baseline": DummyRegressor(strategy="mean"),
    "ridge": Ridge(alpha=1.0, random_state=RANDOM_SEED),
    "sgd": SGDRegressor(random_state=RANDOM_SEED, max_iter=3000, tol=1e-4, penalty="elasticnet", alpha=0.0005),
    "random_forest": RandomForestRegressor(n_estimators=60, min_samples_leaf=3, random_state=RANDOM_SEED, n_jobs=1),
    "gradient_boosting": GradientBoostingRegressor(n_estimators=80, learning_rate=0.06, max_depth=3, random_state=RANDOM_SEED),
    "ann_mlp_40x40": MLPRegressor(hidden_layer_sizes=(40, 40), activation="relu", solver="adam", max_iter=150, random_state=RANDOM_SEED, early_stopping=True, n_iter_no_change=8, validation_fraction=0.15),
}

CLASSIFICATION_MODELS = {
    "majority_baseline": DummyClassifier(strategy="most_frequent"),
    "logistic_balanced": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_SEED),
    "random_forest_cls": RandomForestClassifier(n_estimators=120, min_samples_leaf=3, class_weight="balanced", random_state=RANDOM_SEED, n_jobs=1),
    "gradient_boosting_cls": GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, random_state=RANDOM_SEED),
    "ann_mlp_cls_30x30": MLPClassifier(hidden_layer_sizes=(30, 30), activation="relu", solver="adam", max_iter=200, random_state=RANDOM_SEED, early_stopping=True, n_iter_no_change=10, validation_fraction=0.15),
}


def available_features(train_df: pd.DataFrame) -> list[str]:
    candidates = [c for c in BASE_FEATURES if c in train_df.columns and c not in BLOCKED_FEATURES]
    coverage = train_df[candidates].notna().mean().sort_values(ascending=False) if candidates else pd.Series(dtype=float)
    selected = coverage[coverage >= MIN_TRAIN_FEATURE_COVERAGE].index.tolist()
    return selected


def row_completeness_mask(df: pd.DataFrame, feature_cols: list[str]) -> pd.Series:
    if not feature_cols:
        return pd.Series(False, index=df.index)
    return df[feature_cols].notna().mean(axis=1) >= MIN_ROW_FEATURE_FRACTION


def target_rows(df: pd.DataFrame, target_col: str, wells: list[str], feature_cols: list[str] | None = None) -> pd.DataFrame:
    sub = df[df["well_alias"].isin(wells)].copy()
    sub = sub[sub[target_col].notna()].copy()
    if feature_cols:
        sub = sub[row_completeness_mask(sub, feature_cols)].copy()
    return sub


def equal_well_weights(df: pd.DataFrame) -> np.ndarray:
    counts = df["well_alias"].value_counts().to_dict()
    w = df["well_alias"].map(lambda x: 1.0 / max(counts.get(x, 1), 1)).astype(float).to_numpy()
    return w / np.nanmean(w)


def fit_pipeline(model, X, y, sample_weight=None) -> Pipeline:
    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", MinMaxScaler()),
        ("model", clone(model)),
    ])
    if sample_weight is not None:
        try:
            pipe.fit(X, y, model__sample_weight=sample_weight)
        except Exception:
            pipe.fit(X, y)
    else:
        pipe.fit(X, y)
    return pipe


def metric_dict(y_true, y_pred) -> dict[str, float]:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return {
        "n": int(np.isfinite(y_true).sum()),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(math.sqrt(mean_squared_error(y_true, y_pred))),
        "r2": float(r2_score(y_true, y_pred)) if len(y_true) > 1 else np.nan,
        "bias": float(np.nanmean(y_pred - y_true)),
        "prediction_min": float(np.nanmin(y_pred)),
        "prediction_max": float(np.nanmax(y_pred)),
    }


def classifier_probability(pipe: Pipeline, X: pd.DataFrame) -> np.ndarray:
    model = pipe.named_steps["model"]
    if hasattr(pipe, "predict_proba"):
        proba = pipe.predict_proba(X)
        classes = list(getattr(model, "classes_", []))
        if 1 in classes:
            return proba[:, classes.index(1)]
        return proba[:, -1]
    # Fallback for rare classifiers without probabilities.
    pred = pipe.predict(X)
    return np.asarray(pred, dtype=float)


def classification_metric_dict(y_true, proba, pred_label) -> dict[str, float]:
    y_true = np.asarray(y_true, dtype=int)
    proba = np.asarray(proba, dtype=float)
    pred_label = np.asarray(pred_label, dtype=int)
    out = {
        "n": int(len(y_true)),
        "accuracy": float(accuracy_score(y_true, pred_label)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, pred_label)),
        "precision": float(precision_score(y_true, pred_label, zero_division=0)),
        "recall": float(recall_score(y_true, pred_label, zero_division=0)),
        "f1": float(f1_score(y_true, pred_label, zero_division=0)),
        "brier": float(brier_score_loss(y_true, np.clip(proba, 0, 1))),
        "positive_rate_reference": float(np.mean(y_true == 1)),
        "positive_rate_predicted": float(np.mean(pred_label == 1)),
        "probability_min": float(np.nanmin(proba)),
        "probability_max": float(np.nanmax(proba)),
    }
    out["roc_auc"] = float(roc_auc_score(y_true, proba)) if len(np.unique(y_true)) == 2 else np.nan
    return out


def run_regression_target(
    df: pd.DataFrame,
    target_name: str,
    target_col: str,
    train_wells: list[str],
    validation_well: str,
) -> dict[str, Any]:
    train0 = df[df["well_alias"].isin(train_wells) & df[target_col].notna()].copy()
    if len(train0) < 20:
        raise ValueError(f"Not enough training rows for {target_name}: {len(train0)}")
    features = available_features(train0)
    if not features:
        raise ValueError(f"No features passed coverage threshold for {target_name}")
    train = target_rows(df, target_col, train_wells, features)
    valid = target_rows(df, target_col, [validation_well], features)
    if len(valid) < 5:
        raise ValueError(f"Not enough validation rows for {target_name} on {validation_well}: {len(valid)}")

    X_train, y_train = train[features], train[target_col].astype(float)
    X_valid, y_valid = valid[features], valid[target_col].astype(float)
    weights = equal_well_weights(train)

    metrics_rows = []
    fitted = {}
    prediction_tables = []
    for model_name, model in REGRESSION_MODELS.items():
        print(f"Training {target_name}: {model_name} ...", flush=True)
        pipe = fit_pipeline(model, X_train, y_train, sample_weight=weights)
        pred_raw = pipe.predict(X_valid)
        pred = np.clip(pred_raw, 0.0, 1.0)
        m = metric_dict(y_valid, pred)
        m.update({
            "target": target_name,
            "target_column": target_col,
            "model": model_name,
            "train_wells": "+".join(train_wells),
            "validation_well": validation_well,
            "feature_count": len(features),
            "train_rows": len(train),
            "validation_rows": len(valid),
            "clipped_fraction": float(np.mean((pred_raw < 0) | (pred_raw > 1))),
        })
        metrics_rows.append(m)
        fitted[model_name] = pipe
        tmp = valid[["well_alias", "well_name", "site", "depth_m", "depth_ft", target_col, "hydrate_occurrence_screen", "occurrence_probability_screen", "hydrate_occurrence_label", "occurrence_label_status", "qc_status", "sh_nmr_density_calc", "sh_archie_calc", "sw_archie_calc", "rw_est_ohm_m"]].copy()
        tmp = tmp.rename(columns={target_col: "reference"})
        tmp["target"] = target_name
        tmp["model"] = model_name
        tmp["prediction"] = pred
        tmp["prediction_raw"] = pred_raw
        tmp["residual"] = tmp["prediction"] - tmp["reference"]
        prediction_tables.append(tmp)

    metrics = pd.DataFrame(metrics_rows).sort_values(["rmse", "mae"]).reset_index(drop=True)
    selectable = metrics[~metrics["model"].str.contains("baseline", case=False, na=False)]
    selected_model_name = selectable.iloc[0]["model"] if len(selectable) else metrics.iloc[0]["model"]
    selected_pipe = fitted[selected_model_name]

    all_labeled_wells = sorted(df.loc[df[target_col].notna(), "well_alias"].unique().tolist())
    final_train = target_rows(df, target_col, all_labeled_wells, features)
    final_pipe = fit_pipeline(REGRESSION_MODELS[selected_model_name], final_train[features], final_train[target_col].astype(float), sample_weight=equal_well_weights(final_train))

    try:
        perm = permutation_importance(selected_pipe, X_valid, y_valid, n_repeats=5, random_state=RANDOM_SEED, scoring="neg_root_mean_squared_error")
        imp = pd.DataFrame({
            "target": target_name,
            "model": selected_model_name,
            "feature": features,
            "importance_mean": perm.importances_mean,
            "importance_std": perm.importances_std,
        }).sort_values("importance_mean", ascending=False)
    except Exception as exc:
        imp = pd.DataFrame({"target": [target_name], "model": [selected_model_name], "feature": ["importance_failed"], "importance_mean": [np.nan], "importance_std": [np.nan], "note": [str(exc)]})

    return {
        "task_type": "regression",
        "target": target_name,
        "target_col": target_col,
        "train_wells": train_wells,
        "validation_well": validation_well,
        "features": features,
        "metrics": metrics,
        "predictions": pd.concat(prediction_tables, ignore_index=True),
        "feature_importance": imp,
        "selected_model_name": selected_model_name,
        "selected_validation_pipeline": selected_pipe,
        "final_pipeline": final_pipe,
        "final_train_rows": len(final_train),
        "final_train_wells": all_labeled_wells,
    }


def run_occurrence_classifier(df: pd.DataFrame, train_wells: list[str], validation_well: str) -> dict[str, Any]:
    target_name = "hydrate_occurrence"
    target_col = "hydrate_occurrence_label"
    train0 = df[df["well_alias"].isin(train_wells) & df[target_col].notna()].copy()
    if len(train0) < 20:
        raise ValueError(f"Not enough occurrence-label training rows: {len(train0)}")
    features = available_features(train0)
    if not features:
        raise ValueError("No features passed coverage threshold for occurrence classification")
    train = target_rows(df, target_col, train_wells, features)
    valid = target_rows(df, target_col, [validation_well], features)
    if len(valid) < 5:
        raise ValueError(f"Not enough occurrence validation rows on {validation_well}: {len(valid)}")
    if train[target_col].nunique(dropna=True) < 2:
        raise ValueError("Occurrence classifier needs both positive and negative labels in training wells")

    X_train, y_train = train[features], train[target_col].astype(int)
    X_valid, y_valid = valid[features], valid[target_col].astype(int)
    weights = equal_well_weights(train)

    metrics_rows = []
    fitted = {}
    prediction_tables = []
    for model_name, model in CLASSIFICATION_MODELS.items():
        print(f"Training {target_name}: {model_name} ...", flush=True)
        pipe = fit_pipeline(model, X_train, y_train, sample_weight=weights)
        proba = classifier_probability(pipe, X_valid)
        pred_label = (proba >= OCCURRENCE_CLASSIFICATION_THRESHOLD).astype(int)
        m = classification_metric_dict(y_valid, proba, pred_label)
        m.update({
            "target": target_name,
            "target_column": target_col,
            "model": model_name,
            "train_wells": "+".join(train_wells),
            "validation_well": validation_well,
            "feature_count": len(features),
            "train_rows": len(train),
            "validation_rows": len(valid),
            "positive_threshold_sh": OCCURRENCE_POSITIVE_SH_THRESHOLD,
            "negative_threshold_sh": OCCURRENCE_NEGATIVE_SH_THRESHOLD,
            "classification_threshold": OCCURRENCE_CLASSIFICATION_THRESHOLD,
        })
        metrics_rows.append(m)
        fitted[model_name] = pipe
        tmp = valid[["well_alias", "well_name", "site", "depth_m", "depth_ft", "hydrate_saturation_reference", "hydrate_occurrence_label", "occurrence_label_status", "hydrate_occurrence_screen", "occurrence_probability_screen", "qc_status", "sh_nmr_density_calc", "sh_archie_calc", "sw_archie_calc", "rw_est_ohm_m"]].copy()
        tmp["target"] = target_name
        tmp["model"] = model_name
        tmp["reference"] = y_valid.to_numpy()
        tmp["occurrence_probability_ml"] = np.clip(proba, 0.0, 1.0)
        tmp["predicted_label"] = pred_label
        tmp["label_residual"] = tmp["predicted_label"] - tmp["reference"]
        prediction_tables.append(tmp)

    metrics = pd.DataFrame(metrics_rows)
    selectable = metrics[~metrics["model"].str.contains("baseline", case=False, na=False)].copy()
    if len(selectable):
        selectable["selection_score"] = selectable["roc_auc"].fillna(selectable["f1"]).fillna(selectable["balanced_accuracy"])
        selected_model_name = selectable.sort_values(["selection_score", "f1", "balanced_accuracy"], ascending=False).iloc[0]["model"]
    else:
        selected_model_name = metrics.sort_values(["f1", "balanced_accuracy"], ascending=False).iloc[0]["model"]
    selected_pipe = fitted[selected_model_name]

    all_labeled_wells = sorted(df.loc[df[target_col].notna(), "well_alias"].unique().tolist())
    final_train = target_rows(df, target_col, all_labeled_wells, features)
    final_pipe = fit_pipeline(CLASSIFICATION_MODELS[selected_model_name], final_train[features], final_train[target_col].astype(int), sample_weight=equal_well_weights(final_train))

    try:
        scoring = "roc_auc" if y_valid.nunique() == 2 else "accuracy"
        perm = permutation_importance(selected_pipe, X_valid, y_valid, n_repeats=5, random_state=RANDOM_SEED, scoring=scoring)
        imp = pd.DataFrame({
            "target": target_name,
            "model": selected_model_name,
            "feature": features,
            "importance_mean": perm.importances_mean,
            "importance_std": perm.importances_std,
            "importance_scoring": scoring,
        }).sort_values("importance_mean", ascending=False)
    except Exception as exc:
        imp = pd.DataFrame({"target": [target_name], "model": [selected_model_name], "feature": ["importance_failed"], "importance_mean": [np.nan], "importance_std": [np.nan], "note": [str(exc)]})

    return {
        "task_type": "classification",
        "target": target_name,
        "target_col": target_col,
        "train_wells": train_wells,
        "validation_well": validation_well,
        "features": features,
        "metrics": metrics.sort_values(["f1", "balanced_accuracy"], ascending=False).reset_index(drop=True),
        "predictions": pd.concat(prediction_tables, ignore_index=True),
        "feature_importance": imp,
        "selected_model_name": selected_model_name,
        "selected_validation_pipeline": selected_pipe,
        "final_pipeline": final_pipe,
        "final_train_rows": len(final_train),
        "final_train_wells": all_labeled_wells,
    }

# Run saturation/regression outputs.
results = []
results.append(run_regression_target(features_df, "hydrate_saturation", "hydrate_saturation_reference", HYDRATE_TRAIN_WELLS, HYDRATE_VALIDATION_WELL))
results.append(run_regression_target(features_df, "water_saturation", "water_saturation_reference", WATER_TRAIN_WELLS, WATER_VALIDATION_WELL))

# Run occurrence/classification output from S_h-derived label rule.
occurrence_result = run_occurrence_classifier(features_df, OCCURRENCE_TRAIN_WELLS, OCCURRENCE_VALIDATION_WELL)

metrics_df = pd.concat([r["metrics"] for r in results], ignore_index=True)
predictions_df = pd.concat([r["predictions"] for r in results], ignore_index=True)
occurrence_metrics_df = occurrence_result["metrics"]
occurrence_predictions_df = occurrence_result["predictions"]
feature_importance_df = pd.concat([*[r["feature_importance"] for r in results], occurrence_result["feature_importance"]], ignore_index=True)

selected_rows = []
for r in results:
    best = r["metrics"].loc[r["metrics"]["model"] == r["selected_model_name"]].iloc[0].to_dict()
    selected_rows.append({
        "target": r["target"],
        "task_type": r["task_type"],
        "selected_model": r["selected_model_name"],
        "validation_well": r["validation_well"],
        "train_wells": "+".join(r["train_wells"]),
        "final_train_wells": "+".join(r["final_train_wells"]),
        "features_used": ", ".join(r["features"]),
        "feature_count": len(r["features"]),
        "final_train_rows": r["final_train_rows"],
        "validation_rmse": best["rmse"],
        "validation_mae": best["mae"],
        "validation_r2": best["r2"],
        "validation_accuracy": np.nan,
        "validation_f1": np.nan,
        "validation_roc_auc": np.nan,
        "note": "Water target is limited two-well transfer" if r["target"] == "water_saturation" else "Hydrate saturation fixed WellD holdout",
    })

best_occ = occurrence_result["metrics"].loc[occurrence_result["metrics"]["model"] == occurrence_result["selected_model_name"]].iloc[0].to_dict()
selected_rows.append({
    "target": occurrence_result["target"],
    "task_type": occurrence_result["task_type"],
    "selected_model": occurrence_result["selected_model_name"],
    "validation_well": occurrence_result["validation_well"],
    "train_wells": "+".join(occurrence_result["train_wells"]),
    "final_train_wells": "+".join(occurrence_result["final_train_wells"]),
    "features_used": ", ".join(occurrence_result["features"]),
    "feature_count": len(occurrence_result["features"]),
    "final_train_rows": occurrence_result["final_train_rows"],
    "validation_rmse": np.nan,
    "validation_mae": np.nan,
    "validation_r2": np.nan,
    "validation_accuracy": best_occ["accuracy"],
    "validation_f1": best_occ["f1"],
    "validation_roc_auc": best_occ["roc_auc"],
    "note": f"Occurrence label demo: positive Sh >= {OCCURRENCE_POSITIVE_SH_THRESHOLD:.2f}, negative Sh <= {OCCURRENCE_NEGATIVE_SH_THRESHOLD:.2f}; gray zone omitted",
})
selected_summary_df = pd.DataFrame(selected_rows)

print("Selected models:")
display(selected_summary_df)
print("Regression metrics:")
display(metrics_df.sort_values(["target", "rmse"]))
print("Occurrence classifier metrics:")
display(occurrence_metrics_df)

## 4. Export consolidated outputs

In [ ]:
# =============================================================================
# Consolidated outputs
# =============================================================================

OUTPUT_XLSX = OUTPUT_DIR / "model_results.xlsx"
PREDICTIONS_CSV = OUTPUT_DIR / "predictions.csv"
OCCURRENCE_PREDICTIONS_CSV = OUTPUT_DIR / "occurrence_predictions.csv"
FIGURES_PDF = OUTPUT_DIR / "paper_figures.pdf"
MANIFEST_JSON = OUTPUT_DIR / "run_manifest.json"
MODEL_JOBLIB = MODEL_DIR / "selected_models.joblib"

# Compact summary sheet.
well_summary = features_df.groupby(["well_alias", "well_name", "site"], dropna=False).agg(
    rows=("well_alias", "size"),
    hydrate_target_rows=("hydrate_saturation_reference", lambda x: int(x.notna().sum())),
    occurrence_labeled_rows=("hydrate_occurrence_label", lambda x: int(x.notna().sum())),
    occurrence_positive_rows=("hydrate_occurrence_label", lambda x: int((x == 1).sum())),
    occurrence_negative_rows=("hydrate_occurrence_label", lambda x: int((x == 0).sum())),
    water_target_rows=("water_saturation_reference", lambda x: int(x.notna().sum())),
    rt_coverage=("rt_ohm_m", lambda x: float(x.notna().mean())),
    vp_coverage=("vp_m_s", lambda x: float(x.notna().mean())),
    vs_coverage=("vs_m_s", lambda x: float(x.notna().mean())),
    qc_review_rows=("qc_status", lambda x: int((x == "review").sum())),
    rw_est_ohm_m=("rw_est_ohm_m", lambda x: float(x.dropna().median()) if x.notna().any() else np.nan),
).reset_index()

occ_counts = features_df.groupby(["well_alias", "hydrate_occurrence_screen"], dropna=False).size().reset_index(name="rows")
occ_pivot = occ_counts.pivot(index="well_alias", columns="hydrate_occurrence_screen", values="rows").fillna(0).reset_index()

occ_label_counts = features_df.groupby(["well_alias", "occurrence_label_status", "hydrate_occurrence_label"], dropna=False).size().reset_index(name="rows")
occurrence_label_rule_df = pd.DataFrame([
    {"item": "positive_label", "rule": f"hydrate_occurrence_label = 1 where hydrate_saturation_reference >= {OCCURRENCE_POSITIVE_SH_THRESHOLD:.2f}", "purpose": "Target-side occurrence label for classifier"},
    {"item": "negative_label", "rule": f"hydrate_occurrence_label = 0 where hydrate_saturation_reference <= {OCCURRENCE_NEGATIVE_SH_THRESHOLD:.2f}", "purpose": "Clear non-hydrate / water-like target interval"},
    {"item": "gray_zone", "rule": f"{OCCURRENCE_NEGATIVE_SH_THRESHOLD:.2f} < hydrate_saturation_reference < {OCCURRENCE_POSITIVE_SH_THRESHOLD:.2f} omitted from occurrence training", "purpose": "Avoid teaching classifier ambiguous transition labels"},
    {"item": "leakage_guardrail", "rule": "hydrate_occurrence_label and hydrate_saturation_reference are blocked from X_allowed", "purpose": "Classifier learns from logs/equation features only"},
    {"item": "validation_choice", "rule": f"train {'+'.join(OCCURRENCE_TRAIN_WELLS)} -> validate {OCCURRENCE_VALIDATION_WELL}", "purpose": "Fixed leave-one-well-out demonstration"},
])

qc_summary = features_df.groupby(["well_alias", "qc_status", "qc_caliper_status"], dropna=False).agg(
    rows=("well_alias", "size"),
    low_required_log_coverage_rows=("qc_low_required_log_coverage_flag", lambda x: int(x.fillna(False).sum())),
    missing_rt_rows=("qc_missing_rt_flag", lambda x: int(x.fillna(False).sum())),
    missing_velocity_rows=("qc_missing_velocity_flag", lambda x: int(x.fillna(False).sum())),
    missing_porosity_rows=("qc_missing_porosity_flag", lambda x: int(x.fillna(False).sum())),
    bad_caliper_rows=("qc_bad_caliper_flag", lambda x: int(x.fillna(False).sum())),
    elastic_invalid_rows=("qc_elastic_invalid_flag", lambda x: int(x.fillna(False).sum())),
).reset_index()

run_summary_rows = [
    {"section": "run", "name": "run_id", "value": RUN_ID},
    {"section": "run", "name": "input_dir", "value": str(INPUT_DIR)},
    {"section": "run", "name": "output_dir", "value": str(OUTPUT_DIR)},
    {"section": "run", "name": "model_dir", "value": str(MODEL_DIR)},
    {"section": "policy", "name": "normalized_input_mode", "value": str(NORMALIZED_INPUT_MODE)},
    {"section": "policy", "name": "deep_resistivity_alias_confirmed", "value": str(DEEP_RESISTIVITY_ALIAS_CONFIRMED)},
    {"section": "policy", "name": "hydrate_split", "value": f"{'+'.join(HYDRATE_TRAIN_WELLS)} -> {HYDRATE_VALIDATION_WELL}"},
    {"section": "policy", "name": "occurrence_split", "value": f"{'+'.join(OCCURRENCE_TRAIN_WELLS)} -> {OCCURRENCE_VALIDATION_WELL}"},
    {"section": "policy", "name": "water_split", "value": f"{'+'.join(WATER_TRAIN_WELLS)} -> {WATER_VALIDATION_WELL}"},
    {"section": "policy", "name": "occurrence_label_rule", "value": f"1 if Sh >= {OCCURRENCE_POSITIVE_SH_THRESHOLD:.2f}; 0 if Sh <= {OCCURRENCE_NEGATIVE_SH_THRESHOLD:.2f}; gray zone omitted"},
    {"section": "policy", "name": "rw_policy", "value": "estimate Rw from S_wr/Swr where available; proxy only in normalized-input mode" if ESTIMATE_RW_FROM_WATER_TARGET else "no Rw estimation"},
    {"section": "policy", "name": "nmr_porosity_feature", "value": str(INCLUDE_NMR_POROSITY_AS_FEATURE)},
]
summary_df = pd.concat([
    pd.DataFrame(run_summary_rows),
    selected_summary_df.assign(section="selected_model").rename(columns={"target": "name", "selected_model": "value"})[["section", "name", "value", "task_type", "validation_well", "train_wells", "validation_rmse", "validation_mae", "validation_r2", "validation_accuracy", "validation_f1", "validation_roc_auc", "note"]],
], ignore_index=True, sort=False)

# Add equation constants and Archie metadata.
archie_meta_df = pd.DataFrame([
    {
        "well_alias": alias,
        "well_name": meta["well_name"],
        "site": meta["site"],
        "lat": meta.get("lat", np.nan),
        "lon": meta.get("lon", np.nan),
        "archie_a": meta.get("archie_a", np.nan),
        "archie_m": meta.get("archie_m", np.nan),
        "archie_n": meta.get("archie_n", np.nan),
        "archie_note": meta.get("archie_note", ""),
        "rw_est_ohm_m": features_df.loc[features_df["well_alias"] == alias, "rw_est_ohm_m"].dropna().median() if features_df.loc[features_df["well_alias"] == alias, "rw_est_ohm_m"].notna().any() else np.nan,
        "hydrate_target_header": meta.get("hydrate_target_header"),
        "water_target_header": meta.get("water_target_header"),
    }
    for alias, meta in WELL_METADATA.items()
])
equations_compact = pd.concat([
    equations_used,
    pd.DataFrame([
        {"field": "density_constants", "equation": "rho_matrix=2.65 g/cc; rho_fluid=1.02 g/cc", "role": "assumption", "predictor_allowed": "n/a", "notes": "Used for density-porosity calculation; proxy caveat applies in normalized-input mode."},
        {"field": "fixed_welld_holdout", "equation": f"hydrate: {'+'.join(HYDRATE_TRAIN_WELLS)} -> {HYDRATE_VALIDATION_WELL}; occurrence: {'+'.join(OCCURRENCE_TRAIN_WELLS)} -> {OCCURRENCE_VALIDATION_WELL}; water: {'+'.join(WATER_TRAIN_WELLS)} -> {WATER_VALIDATION_WELL}", "role": "validation design", "predictor_allowed": "n/a", "notes": "Single fixed leave-one-well-out choice for mentor review."},
        {"field": "deep_resistivity_alias", "equation": ", ".join(DEEP_RESISTIVITY_ALIASES_CONFIRMED), "role": "data dictionary", "predictor_allowed": "n/a", "notes": "Confirmed by project review as deep formation resistivity family."},
    ])
], ignore_index=True)

# Write predictions.
predictions_df.to_csv(PREDICTIONS_CSV, index=False)
occurrence_predictions_df.to_csv(OCCURRENCE_PREDICTIONS_CSV, index=False)

# Write figures.
with PdfPages(FIGURES_PDF) as pdf:
    # 1. Saturation prediction vs reference.
    for r in results:
        target = r["target"]
        selected = r["selected_model_name"]
        plot_df = predictions_df[(predictions_df["target"] == target) & (predictions_df["model"] == selected)].copy()
        if len(plot_df):
            fig, ax = plt.subplots(figsize=(6.5, 5.5))
            ax.scatter(plot_df["reference"], plot_df["prediction"], s=12, alpha=0.7)
            ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1)
            ax.set_xlim(0, 1)
            ax.set_ylim(0, 1)
            ax.set_xlabel("Reference saturation")
            ax.set_ylabel("Predicted saturation")
            m = metric_dict(plot_df["reference"], plot_df["prediction"])
            ax.set_title(f"{target}: {selected}\n{r['train_wells']} → {r['validation_well']} | RMSE={m['rmse']:.3f}, R²={m['r2']:.3f}, n={m['n']}")
            ax.grid(True, alpha=0.25)
            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)

            fig, ax = plt.subplots(figsize=(7, 7))
            depth = plot_df["depth_m"]
            ax.plot(plot_df["reference"], depth, label="reference", linewidth=1.8)
            ax.plot(plot_df["prediction"], depth, label="prediction", linewidth=1.4)
            ax.set_xlim(0, 1)
            ax.invert_yaxis()
            ax.set_xlabel("Saturation fraction")
            ax.set_ylabel("Depth (m)")
            ax.set_title(f"Depth profile — {target}, {r['validation_well']} ({selected})")
            ax.legend()
            ax.grid(True, alpha=0.25)
            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)

    # 2. Regression model comparison.
    fig, ax = plt.subplots(figsize=(8, 5))
    comp = metrics_df.copy().sort_values(["target", "rmse"])
    labels = comp["target"].str.replace("_", " ") + "\n" + comp["model"]
    ax.bar(np.arange(len(comp)), comp["rmse"])
    ax.set_xticks(np.arange(len(comp)))
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_ylabel("Blind-well RMSE")
    ax.set_title("Saturation regression model comparison")
    ax.grid(True, axis="y", alpha=0.25)
    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)

    # 3. Rule-based occurrence screen counts.
    fig, ax = plt.subplots(figsize=(8, 5))
    occ_plot = occ_counts.copy()
    for label, sub in occ_plot.groupby("hydrate_occurrence_screen"):
        ax.bar(sub["well_alias"], sub["rows"], label=label, alpha=0.75)
    ax.set_ylabel("Rows")
    ax.set_title("Rule-based hydrate occurrence screen counts")
    ax.legend(fontsize=8)
    ax.grid(True, axis="y", alpha=0.25)
    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)

    # 4. Occurrence label rule counts.
    fig, ax = plt.subplots(figsize=(8, 5))
    label_plot = occ_label_counts.copy()
    for status, sub in label_plot.groupby("occurrence_label_status"):
        ax.bar(sub["well_alias"], sub["rows"], label=status, alpha=0.75)
    ax.set_ylabel("Rows")
    ax.set_title("Occurrence label rule counts from S_h target")
    ax.legend(fontsize=7)
    ax.grid(True, axis="y", alpha=0.25)
    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)

    # 5. Occurrence classifier probability by depth for selected model.
    selected_occ = occurrence_result["selected_model_name"]
    occ_pred_plot = occurrence_predictions_df[occurrence_predictions_df["model"] == selected_occ].copy()
    if len(occ_pred_plot):
        fig, ax = plt.subplots(figsize=(7, 7))
        ax.plot(occ_pred_plot["occurrence_probability_ml"], occ_pred_plot["depth_m"], label="ML occurrence probability", linewidth=1.4)
        ax.scatter(occ_pred_plot["reference"], occ_pred_plot["depth_m"], s=10, alpha=0.5, label="S_h label")
        ax.set_xlim(0, 1)
        ax.invert_yaxis()
        ax.set_xlabel("Occurrence probability / label")
        ax.set_ylabel("Depth (m)")
        ax.set_title(f"Occurrence classifier — {OCCURRENCE_VALIDATION_WELL} ({selected_occ})")
        ax.legend()
        ax.grid(True, alpha=0.25)
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

# Write compact Excel workbook. If Excel has the file open, write timestamp fallback.
def write_excel(path: Path) -> Path:
    sheets = {
        "summary": summary_df,
        "metrics": metrics_df.sort_values(["target", "rmse"]),
        "occurrence_ml_metrics": occurrence_metrics_df,
        "equations_used": equations_compact,
        "archie_metadata": archie_meta_df,
        "feature_importance": feature_importance_df,
        "occurrence_screen": occ_pivot,
        "occurrence_label_rule": pd.concat([occurrence_label_rule_df, occ_label_counts.rename(columns={"occurrence_label_status": "rule", "well_alias": "item"}).assign(purpose="label count by well")], ignore_index=True, sort=False),
        "occurrence_ml_preds": occurrence_predictions_df,
        "qc_summary": qc_summary,
        "well_summary": well_summary,
        "field_mapping": field_mapping,
    }
    try:
        with pd.ExcelWriter(path, engine="openpyxl") as writer:
            for sheet_name, df_sheet in sheets.items():
                df_sheet.to_excel(writer, sheet_name=sheet_name[:31], index=False)
        return path
    except PermissionError:
        fallback = path.with_name(f"{path.stem}_{RUN_ID}{path.suffix}")
        with pd.ExcelWriter(fallback, engine="openpyxl") as writer:
            for sheet_name, df_sheet in sheets.items():
                df_sheet.to_excel(writer, sheet_name=sheet_name[:31], index=False)
        return fallback

actual_xlsx = write_excel(OUTPUT_XLSX)

# Save selected models and config.
model_bundle = {
    "run_id": RUN_ID,
    "well_metadata": WELL_METADATA,
    "normalized_input_mode": NORMALIZED_INPUT_MODE,
    "deep_resistivity_alias_confirmed": DEEP_RESISTIVITY_ALIAS_CONFIRMED,
    "occurrence_label_rule": {
        "positive_sh_threshold": OCCURRENCE_POSITIVE_SH_THRESHOLD,
        "negative_sh_threshold": OCCURRENCE_NEGATIVE_SH_THRESHOLD,
        "gray_zone_omitted": True,
    },
    "selected_models": {
        r["target"]: {
            "task_type": r["task_type"],
            "selected_model_name": r["selected_model_name"],
            "features": r["features"],
            "final_pipeline": r["final_pipeline"],
            "validation_pipeline": r["selected_validation_pipeline"],
            "train_wells": r["train_wells"],
            "validation_well": r["validation_well"],
        }
        for r in results + [occurrence_result]
    },
}
joblib.dump(model_bundle, MODEL_JOBLIB)

manifest = {
    "run_id": RUN_ID,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "input_dir": str(INPUT_DIR),
    "output_dir": str(OUTPUT_DIR),
    "model_dir": str(MODEL_DIR),
    "model_results_xlsx": str(actual_xlsx),
    "predictions_csv": str(PREDICTIONS_CSV),
    "occurrence_predictions_csv": str(OCCURRENCE_PREDICTIONS_CSV),
    "paper_figures_pdf": str(FIGURES_PDF),
    "selected_models_joblib": str(MODEL_JOBLIB),
    "normalized_input_mode": NORMALIZED_INPUT_MODE,
    "deep_resistivity_alias_confirmed": DEEP_RESISTIVITY_ALIAS_CONFIRMED,
    "hydrate_split": {"train_wells": HYDRATE_TRAIN_WELLS, "validation_well": HYDRATE_VALIDATION_WELL},
    "occurrence_split": {"train_wells": OCCURRENCE_TRAIN_WELLS, "validation_well": OCCURRENCE_VALIDATION_WELL},
    "water_split": {"train_wells": WATER_TRAIN_WELLS, "validation_well": WATER_VALIDATION_WELL},
    "occurrence_label_rule": {
        "positive_sh_threshold": OCCURRENCE_POSITIVE_SH_THRESHOLD,
        "negative_sh_threshold": OCCURRENCE_NEGATIVE_SH_THRESHOLD,
        "classification_threshold": OCCURRENCE_CLASSIFICATION_THRESHOLD,
    },
    "selected_models": selected_summary_df.to_dict(orient="records"),
}
MANIFEST_JSON.write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")

print("\nDONE — files written:")
for p in [actual_xlsx, PREDICTIONS_CSV, OCCURRENCE_PREDICTIONS_CSV, FIGURES_PDF, MANIFEST_JSON, MODEL_JOBLIB]:
    print(" -", p, "exists=", p.exists(), "modified=", datetime.fromtimestamp(p.stat().st_mtime) if p.exists() else None)